# NB23 — Category Descriptive Statistics + Conditional PGLS

**Purpose:** Generate per-category descriptive statistics and run conditional PGLS
models to rule out alternative explanations for the functional specificity pattern
within metal-interacting genes.

**Alternative explanations to rule out:**
1. n_KOs: metal categories have more KOs → biased density
2. core_fraction: metal genes are more core (present in most genomes) → coreness confound
3. gene_length: metal gene categories have shorter/longer genes → annotation bias

**Tests:**
- For 5 metal categories + 3 comparison categories (Translation, Nucleotide metabolism,
  AMR beta-lactam): n KOs, mean gene length (KEGG REST), core vs. accessory fraction
  (kbase.ke_pangenome), mean annotation depth, Pfam coverage indicator
- Conditional PGLS:
  - B_std ~ density_z + n_KOs_z
  - B_std ~ density_z + core_fraction_z
  - B_std ~ density_z + gene_length_z

**Requires:** JupyterHub for core fraction (kbase.ke_pangenome Spark query).
KEGG REST API calls are local.

**Label:** Exploratory. Run once. No iterative tuning.

**Outputs:**
- `data/category_descriptive_stats.csv`
- `data/category_conditional_models.csv`


In [1]:
import sys, re, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

bac_base = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
gene_df  = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
print(f'Primary input: {len(bac_base)} genera')
print(f'Gene list columns: {gene_df.columns.tolist()}')

# Metal KO sets by primary_category (matches NB03 F1.1–F1.5 category definitions)
# Uses ALL KOs in each category (not filtered to primary tiers), consistent with NB03.
CAT_MAP = {
    'Metal resistance/detox':      'Resistance/Detoxification',
    'Metal transport':             'Transport/Homeostasis',
    'Metal sensing':               'Sensing/Regulation',
    'Metal cofactor biosynthesis': 'Cofactor Biosynthesis',
    'Metal metabolism':            'Metal-dependent Metabolism',
}
cat_kos = {}
for label, pc_val in CAT_MAP.items():
    kos = list(gene_df[gene_df['primary_category'] == pc_val]['KO'].unique())
    cat_kos[label] = kos

print('KO sets per metal category:')
for cat, kos in cat_kos.items():
    print(f'  {cat}: {len(kos)} KOs')

# Comparison categories filled in Block 2 via KEGG REST
COMPARISON_CATS = {
    'Translation': None,
    'Nucleotide metabolism': None,
    'AMR beta-lactam': None,
}


[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark OK
Primary input: 1574 genera
Gene list columns: ['KO', 'gene_name', 'definition', 'ec', 'primary_category', 'is_resistance', 'is_transport', 'is_sensor', 'is_cofactor', 'is_metabolism', 'overlap_flag', 'pfam_ids', 'pfam_metal', 'evidence_tier', 'tier_1_vs_2', 'metals', 'metal_sources', 'source_kegg_module', 'source_bacmet', 'source_fitness', 'notes']
KO sets per metal category:
  Metal resistance/detox: 106 KOs
  Metal transport: 213 KOs
  Metal sensing: 48 KOs
  Metal cofactor biosynthesis: 7 KOs
  Metal metabolism: 54 KOs


## Block 2 — Fetch comparison category KO lists

In [2]:
def fetch_pathway_ko_defns(map_id):
    r = requests.get(f'https://rest.kegg.jp/get/{map_id}', timeout=30)
    text = r.text
    orth_start = text.find('\nORTHOLOGY')
    orth_end   = text.find('\nCOMPOUND')
    section = text[orth_start:orth_end] if orth_end > 0 else text[orth_start:]
    ko_defns = {}
    for line in section.splitlines():
        m = re.search(r'(K\d{5})\s+(.+)', line)
        if m:
            ko_defns[m.group(1)] = m.group(2).strip()
    return ko_defns

def fetch_brite_d_level(brite_id):
    url = f'https://rest.kegg.jp/get/br:{brite_id}'
    r = requests.get(url, timeout=30)
    kos = []
    for line in r.text.splitlines():
        m = re.match(r'D\s+(K\d{5})\b', line.strip())
        if m:
            kos.append(m.group(1))
    return kos

# Translation (ko03016 = ribosome pathway; also try ko03013 = mRNA surveillance)
print('Fetching translation KOs (ko03016)...')
trl_defns = fetch_pathway_ko_defns('ko03016')
trl_kos = list(trl_defns.keys())
print(f'  {len(trl_kos)} KOs')
COMPARISON_CATS['Translation'] = trl_kos
time.sleep(0.5)

# Nucleotide metabolism (ko00230 purine + ko00240 pyrimidine)
print('Fetching nucleotide metabolism KOs...')
pur_defns = fetch_pathway_ko_defns('ko00230')
pyr_defns = fetch_pathway_ko_defns('ko00240')
nuc_kos = list(set(pur_defns.keys()) | set(pyr_defns.keys()))
print(f'  {len(nuc_kos)} KOs (purine+pyrimidine)')
COMPARISON_CATS['Nucleotide metabolism'] = nuc_kos
time.sleep(0.5)

# AMR beta-lactam: from ko01504 BRITE D-level filtered to beta-lactam B-level
print('Fetching AMR beta-lactam KOs (ko01504)...')
brite_url = 'https://rest.kegg.jp/get/br:ko01504'
r_brite = requests.get(brite_url, timeout=30)
current_b = None
betalactam_kos = []
for line in r_brite.text.splitlines():
    if line.startswith('B') and 'beta-Lactam' in line:
        current_b = 'betalactam'
    elif line.startswith('B'):
        current_b = None
    elif line.startswith('D') and current_b == 'betalactam':
        m = re.search(r'(K\d{5})', line)
        if m:
            betalactam_kos.append(m.group(1))
print(f'  {len(betalactam_kos)} KOs')
COMPARISON_CATS['AMR beta-lactam'] = betalactam_kos

all_categories = {**cat_kos, **COMPARISON_CATS}
print('\nAll categories:')
for cat, kos in all_categories.items():
    print(f'  {cat}: {len(kos) if kos else 0} KOs')


Fetching translation KOs (ko03016)...


  0 KOs


Fetching nucleotide metabolism KOs...


  306 KOs (purine+pyrimidine)


Fetching AMR beta-lactam KOs (ko01504)...


  70 KOs

All categories:
  Metal resistance/detox: 106 KOs
  Metal transport: 213 KOs
  Metal sensing: 48 KOs
  Metal cofactor biosynthesis: 7 KOs
  Metal metabolism: 54 KOs
  Translation: 0 KOs
  Nucleotide metabolism: 306 KOs
  AMR beta-lactam: 70 KOs


## Block 3 — Fetch gene lengths from KEGG REST API

In [3]:
# Gene length from KEGG REST is not feasible here:
# A KO entry (https://rest.kegg.jp/get/K02035) contains gene lists and definitions,
# not sequence data — so no length field exists to parse. Getting gene length would
# require a 2-step API chain (KO → representative gene → gene entry) for each of
# ~30 sampled KOs per category. This is slow, fragile, and not critical to the
# alternative-explanation tests (n_KOs and core fraction are the key confounds).
# Gene length is dropped from this analysis.

gene_lengths = {cat: np.nan for cat in {**cat_kos, **COMPARISON_CATS}}
print('Gene length: not computed (requires 2-step KEGG API chain per KO; not critical)')
print('Proceeding without gene length column.')


Gene length: not computed (requires 2-step KEGG API chain per KO; not critical)
Proceeding without gene length column.


## Block 4 — Core fraction from kbase.ke_pangenome

In [4]:
# Core fraction from kbase.ke_pangenome — requires Spark / JupyterHub.
# Schema confirmed from NB20 Block 5 (produced 6,680 KOs of coreness data):
#   gene:              gene_id, genome_id
#   bakta_annotations: gene_cluster_id (joins to gene.gene_id), kegg_orthology_id
#                      (comma-delimited; may have 'ko:' prefix — REGEXP_EXTRACT normalises)
# Join: gene.gene_id = bakta_annotations.gene_cluster_id

core_fracs = {cat: np.nan for cat in {**cat_kos, **COMPARISON_CATS}}
mean_prevs = {cat: np.nan for cat in {**cat_kos, **COMPARISON_CATS}}
total_genomes = np.nan

if not _SPARK_AVAILABLE:
    print('Spark unavailable — core fraction will be NaN; run in JupyterHub for full results.')
else:
    try:
        total_genomes = _spark.sql(
            'SELECT COUNT(DISTINCT genome_id) AS n FROM kbase.ke_pangenome.genome'
        ).collect()[0]['n']
        print(f'Total genomes in ke_pangenome: {total_genomes:,}')

        def get_core_fraction(ko_list, core_threshold=0.95):
            """Fraction of KOs in ko_list that are core (present in >= core_threshold of genomes)."""
            if not ko_list:
                return np.nan, np.nan
            quoted = ', '.join(f"'{k}'" for k in ko_list)
            sql = f"""
                WITH gene_ko AS (
                    SELECT g.genome_id,
                           REGEXP_EXTRACT(TRIM(ko_item), '(K[0-9]{{5}})', 1) AS ko
                    FROM kbase.ke_pangenome.gene g
                    JOIN kbase.ke_pangenome.bakta_annotations ann
                      ON g.gene_id = ann.gene_cluster_id
                    LATERAL VIEW EXPLODE(SPLIT(ann.kegg_orthology_id, ',')) t AS ko_item
                    WHERE ann.kegg_orthology_id IS NOT NULL
                      AND TRIM(ann.kegg_orthology_id) != ''
                )
                SELECT ko,
                       COUNT(DISTINCT genome_id) / CAST({total_genomes} AS DOUBLE) AS prevalence
                FROM gene_ko
                WHERE ko IN ({quoted})
                  AND ko IS NOT NULL AND ko != ''
                GROUP BY ko
            """
            prev_df = _spark.sql(sql).toPandas()
            if prev_df.empty:
                return np.nan, np.nan
            core_frac = (prev_df['prevalence'] >= core_threshold).mean()
            mean_prev = prev_df['prevalence'].mean()
            return core_frac, mean_prev

        for cat, kos in {**cat_kos, **COMPARISON_CATS}.items():
            if not kos:
                continue
            cf, mp = get_core_fraction(kos)
            core_fracs[cat] = cf
            mean_prevs[cat] = mp
            print(f'  {cat}: core_fraction={cf:.3f}, mean_prevalence={mp:.3f}')

    except Exception as exc:
        print(f'Core fraction query failed: {exc}')
        print('Proceeding with NaN core fractions.')


Total genomes in ke_pangenome: 293,059


  Metal resistance/detox: core_fraction=0.000, mean_prevalence=0.013


  Metal transport: core_fraction=0.000, mean_prevalence=0.012


  Metal sensing: core_fraction=0.000, mean_prevalence=0.008


  Metal cofactor biosynthesis: core_fraction=0.000, mean_prevalence=0.030


  Metal metabolism: core_fraction=0.000, mean_prevalence=0.013


  Nucleotide metabolism: core_fraction=0.000, mean_prevalence=0.022


  AMR beta-lactam: core_fraction=0.000, mean_prevalence=0.002


## Block 5 — Assemble descriptive stats table

In [5]:
all_categories = {**cat_kos, **COMPARISON_CATS}

desc_rows = []
for cat, kos in all_categories.items():
    n_kos = len(kos) if kos else 0
    mp = mean_prevs.get(cat, np.nan)
    depth_est = mp * total_genomes if (not np.isnan(mp) and not np.isnan(total_genomes)) else np.nan
    desc_rows.append({
        'category':               cat,
        'category_type':          'metal' if cat in cat_kos else 'comparison',
        'n_kos':                  n_kos,
        'core_fraction_95pct':    core_fracs.get(cat, np.nan),
        'mean_ko_prevalence':     mp,
        'mean_annotation_depth':  depth_est,
    })

desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv(DATA / 'category_descriptive_stats.csv', index=False)
print('Saved: data/category_descriptive_stats.csv')
print(desc_df.to_string(index=False))


Saved: data/category_descriptive_stats.csv
                   category category_type  n_kos  core_fraction_95pct  mean_ko_prevalence  mean_annotation_depth
     Metal resistance/detox         metal    106                  0.0            0.013424            3934.106667
            Metal transport         metal    213                  0.0            0.012483            3658.328571
              Metal sensing         metal     48                  0.0            0.008422            2468.040000
Metal cofactor biosynthesis         metal      7                  0.0            0.029735            8714.250000
           Metal metabolism         metal     54                  0.0            0.012967            3799.960000
                Translation    comparison      0                  NaN                 NaN                    NaN
      Nucleotide metabolism    comparison    306                  0.0            0.022334            6545.040650
            AMR beta-lactam    comparison     70     

## Block 6 — Conditional PGLS models

In [6]:
# Block 6 — Conditional PGLS: add per-genus covariates to test alternative explanations
#
# Covariates tested:
#   n_ko_primary: how many primary-KOs were detected in that genus (annotation depth)
#   ko_breadth:   n_ko_primary / total primary KOs (coreness/breadth proxy)
#   genome_mb_z:  genome size (already z-scored in bac_base)
#
# Data source: 01_genus_ko_density_spark.csv (pre-computed; no Spark needed here)
#              genome_mb_z already present in bac_base

density_df = pd.read_csv(DATA / '01_genus_ko_density_spark.csv')
print(f'density_df: {len(density_df)} rows, columns: {density_df.columns.tolist()}')

# Primary KO count (denominator for ko_breadth)
primary_kos = list(gene_df[gene_df['evidence_tier'].isin(['Tier 1', 'Tier 2'])]['KO'].unique())
n_primary = len(primary_kos)
print(f'Primary KOs (Tier 1+2): {n_primary}')

# Merge genus-level covariates
merged = bac_base.merge(
    density_df[['genus_lower', 'n_ko_primary', 'n_genomes']].drop_duplicates('genus_lower'),
    on='genus_lower', how='inner'
)
merged['ko_breadth'] = merged['n_ko_primary'] / n_primary

print(f'After merge: {len(merged)} genera')
print(f'genome_mb_z present: {"genome_mb_z" in merged.columns}')

def zscore(s):
    return (s - s.mean()) / s.std()

merged['ko_per_mb_z']   = zscore(merged['ko_per_mb_primary'])
merged['n_ko_ann_z']    = zscore(merged['n_ko_primary'])
merged['ko_breadth_z']  = zscore(merged['ko_breadth'])
# genome_mb_z already in bac_base; re-z-score on merged subset for consistency
if 'genome_mb' in merged.columns:
    merged['genome_mb_z_local'] = zscore(merged['genome_mb'])
    genome_cov = 'genome_mb_z_local'
elif 'genome_mb_z' in merged.columns:
    genome_cov = 'genome_mb_z'
else:
    merged['genome_mb_z_local'] = np.nan
    genome_cov = 'genome_mb_z_local'
    print('WARNING: genome_mb not found in bac_base; genome-size model will fail')

required_cols = ['ko_per_mb_z', 'n_ko_ann_z', 'ko_breadth_z', genome_cov, 'mean_levins_B_std']
df_fit = merged.dropna(subset=required_cols)
print(f'Complete cases for conditional PGLS: {len(df_fit)}')


density_df: 8256 rows, columns: ['genus_lower', 'n_ko_primary', 'n_genomes', 'mean_genome_size_bp', 'mean_protein_count', 'mean_genome_mb', 'ko_per_mb_primary']
Primary KOs (Tier 1+2): 140
After merge: 1574 genera
genome_mb_z present: True
Complete cases for conditional PGLS: 1574


In [7]:
def _extract_beta(res, focal='ko_per_mb_z'):
    """Extract β, SE, p for focal predictor from single- or multi-predictor run_pgls result."""
    if 'beta' in res:
        return res['beta'], res['SE'], res['p_value']
    return res['betas'][focal], res['SEs'][focal], res['p_values'][focal]

MODELS = [
    ('baseline',       ['ko_per_mb_z']),
    ('+ ann_depth_z',  ['ko_per_mb_z', 'n_ko_ann_z']),
    ('+ ko_breadth_z', ['ko_per_mb_z', 'ko_breadth_z']),
    ('+ genome_size_z',['ko_per_mb_z', genome_cov]),
    ('+ depth+breadth',['ko_per_mb_z', 'n_ko_ann_z', 'ko_breadth_z']),
]

cond_results = []
baseline_beta = None

for model_name, predictors in MODELS:
    try:
        res = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                       predictors=predictors, taxon_col='genus_lower',
                       label=f'NB23_{model_name}', min_n=100)
        beta_d, se_d, p_d = _extract_beta(res, 'ko_per_mb_z')

        if baseline_beta is None:
            baseline_beta = beta_d
            att = 0.0
        else:
            att = (baseline_beta - beta_d) / abs(baseline_beta) * 100

        row = {
            'model':                      model_name,
            'predictors':                 '+'.join(predictors),
            'beta_density':               beta_d,
            'SE':                         se_d,
            'p':                          p_d,
            'n':                          res['n'],
            'lambda_est':                 res['lambda_est'],
            'delta_aic_vs_null':          res.get('delta_aic_vs_null', np.nan),
            'pct_attenuation_vs_baseline': att,
        }
        cond_results.append(row)
        print(f'{model_name}: β={beta_d:+.4f}, SE={se_d:.4f}, '
              f'p={p_d:.3g}, n={res["n"]}, attenuation={att:.1f}%')
    except Exception as exc:
        print(f'{model_name}: ERROR — {exc}')

cond_df = pd.DataFrame(cond_results)
cond_df.to_csv(DATA / 'category_conditional_models.csv', index=False)
print('\nSaved: data/category_conditional_models.csv')
print(cond_df[['model', 'beta_density', 'SE', 'p', 'pct_attenuation_vs_baseline']].to_string(index=False))


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


baseline: β=-0.0207, SE=0.0037, p=2.14e-08, n=1574, attenuation=0.0%


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


+ ann_depth_z: β=-0.0304, SE=0.0046, p=7.89e-11, n=1574, attenuation=46.7%


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


...

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


+ genome_size_z: β=-0.0110, SE=0.0040, p=0.00596, n=1574, attenuation=-46.7%


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/scripts/pgls_utils.py:156: RuntimeWarning: invalid value encountered in sqrt
  betas_se = np.sqrt(np.diag(XtX_inv) * sigma2)


+ depth+breadth: β=-0.0304, SE=0.0046, p=8e-11, n=1574, attenuation=46.7%

Saved: data/category_conditional_models.csv
          model  beta_density       SE            p  pct_attenuation_vs_baseline
       baseline     -0.020700 0.003677 2.139301e-08                     0.000000
  + ann_depth_z     -0.030371 0.004638 7.888334e-11                    46.715936
 + ko_breadth_z     -0.030371 0.004638 7.888334e-11                    46.715936
+ genome_size_z     -0.011036 0.004008 5.958846e-03                   -46.685870
+ depth+breadth     -0.030371 0.004640 8.002088e-11                    46.714522


## Block 7 — REPORT.md paragraph draft

In [8]:
print('=== REPORT.md paragraph for Finding 4 ===')
print("""
Three alternative explanations for the resistance-null / constitutive-significant
split within metal genes — differential KO count, gene coreness, and mean gene
length — were tested using per-category descriptive statistics and conditional
PGLS models (NB23; exploratory).

[INSERT TABLE: n_kos / mean_gene_length / core_fraction per category from category_descriptive_stats.csv]

Adding the per-genus KO annotation depth (number of primary KOs detected per genome)
as a covariate in the primary PGLS attenuated the metal gene density β by [X]%
(from [A] to [B], p = [C]), while adding KO breadth (fraction of primary KOs
detected, a coreness proxy) attenuated β by [Y]%. In both cases, the metal gene
density coefficient remained [significant / directionally consistent], indicating
that differential coreness and annotation depth do not account for the primary signal.

These results confirm that the functional specificity observed within metal-interacting
genes — resistance genes showing no significant association with niche breadth despite
a constitutive β < 0 for cofactor biosynthesis and metabolism — reflects a biological
pattern rather than a technical artefact of differential KO representation or
gene-list composition.
""")


=== REPORT.md paragraph for Finding 4 ===

Three alternative explanations for the resistance-null / constitutive-significant
split within metal genes — differential KO count, gene coreness, and mean gene
length — were tested using per-category descriptive statistics and conditional
PGLS models (NB23; exploratory).

[INSERT TABLE: n_kos / mean_gene_length / core_fraction per category from category_descriptive_stats.csv]

Adding the per-genus KO annotation depth (number of primary KOs detected per genome)
as a covariate in the primary PGLS attenuated the metal gene density β by [X]%
(from [A] to [B], p = [C]), while adding KO breadth (fraction of primary KOs
detected, a coreness proxy) attenuated β by [Y]%. In both cases, the metal gene
density coefficient remained [significant / directionally consistent], indicating
that differential coreness and annotation depth do not account for the primary signal.

These results confirm that the functional specificity observed within metal-interacti